In [ ]:
import re
import json
import pandas as pd
import numpy as np
from datetime import datetime
import pycountry
from IPython.display import display, Markdown
from numpyro import distributions as dist
import yaml as yml
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
import git
import yaml as yml

from emu_renewal.constants import BASE_PATH, OUTPUTS_PATH
from emu_renewal.selection import get_mob_avail_countries, gather_who_data, find_absent_inds, \
    find_neg_inds, find_outliers, find_nans_repeats
from emu_renewal.inputs import get_worldbank_national_pop, get_income_group, get_fb_visited_mobility, \
    get_fb_singletile_mobility, process_raw_google_mobility, get_google_mobility, get_country_pop, \
    get_undesa_national_pop, get_country_vacc_data, get_world_shp
from emu_renewal.outputs import get_table_df_from_priors_dict, add_bool_row_to_table, add_mob_avail_to_world
from emu_renewal.document import get_func_blurb
from emu_renewal.indicators import get_deaths_target, get_cases_target, filter_seroprev, get_owid_hosps, \
    get_all_seroprev, get_seroprev_pooled_totals, get_var_target, extract_specific_var, get_country_vars, \
    get_alpha_info, get_delta_info, get_specific_var_props, \
    get_ba2_info, get_ba5_info
from emu_renewal.run import find_run_start_time, find_run_end_time, get_hosp_target, get_seroprev_target, \
    get_mobility_provider, run_calibration
from emu_renewal.geospatial import FacebookMobilityBuilder
from emu_renewal.renew import MultiStrainModel
from emu_renewal.distributions import GammaDens
from emu_renewal.process import _get_cos_curve_at_x
from emu_renewal.mobility import NoMobilityProvider, WeightedExpMobilityProvider, SingleSeriesExpMobilityProvider
from emu_renewal.calibration import custom_init, StandardCalib
from emu_renewal.priors import get_standard_priors
from emu_renewal.plotting import plot_beta_priors, plot_duration_params, plot_inclusion

plt.style.use("default")
set_matplotlib_formats("svg")

{{< pagebreak >}}

# Documentation

In [ ]:
repo = git.Repo(search_parent_directories=True)
commit_id = repo.git.rev_parse("--short", "HEAD")
txt = (
    "The following document was documented algorithmically "
    "from the code used in the analysis.\n"
)
txt += f"This version was produced from commit {commit_id}. \n\n"
txt += "# Analysis methods \n\n"
display(Markdown(txt))

## Summary equation
The mechanics of the renewal process can be summarised in the following equations, the detail of which is expanded in the following sections.
$$i_{p,v,t}=n_{p,t}x_{p,v}(1-e^{-\lambda_{v,t}})$$
$$\lambda_{v,t}=\beta\sigma_{v}e^{W_{t}}M_{t}\sum_{\tau=t-50}^{t-1}{(s_{v,\tau}+\sum_{p'}i_{p',v,\tau})g_{t-\tau}}$$
$$n_{p,t}=n_{p,t-1}-\sum_{v}i_{p,v,t}+i_{\tilde{p},\tilde{v},t}$$
$$M_{t}=
\begin{cases}
1 & \text{if no mobility} \\
f + (\frac{\sum_{l}w_{l}G_{l,t}}{\sum_{l}w_{l}})^{m} \times(1-f) & \text{if Google scaling} \\
f + F_{t}^{m} \times(1-f) & \text{if Facebook scaling} \\
f + (\frac{\sum_{p}w_{p}O_{p,t}}{\sum_{p}w_{p}})\times(1-f) & \text{if OxCGRT scaling} \\
\end{cases}
$$
$$o_{t}=a\sum_{v}\sum_{p}\sum_{\tau=1}^{50}i_{p,v,t-\tau} \times c_{\tau}$$
$$h_{t}=\sum_{\tau=1}^{50}o_{t-\tau}(1-\sum_{u=0}^{\tau-1}d_{u})$$
Where:

- $i_{p,v,t}$ represents the incidence of variant $v$ in immunity status group $p$ at time $t$, where $p$ represents the immunity status prior to infection with variant $v$
- $p\in \{0, 1\}^{V}$ are the possible combination of past variant exposure statuses with $V$ variants
- $n_{p,t}$ is the size of the population with immunity status $p$ at time $t$
- $x_{p,v}$ is the relative susceptibility of persons with immunity status $p$ to infection with variant $v$
- $x_{\{0\}^{V},v} = 1$, full susceptibility for those never previously infected
- $x_{p,v} = 0$ for infection with a variant to which the person has previously been infected with or a preceding variant (e.g. infection with Alpha following infection with Delta)
- $x_{p,v}$ for immunity statuses other than the two preceding cases is the complement of a single calibrated cross-immunity parameter with a beta-distributed prior
- $\beta$ is the infectiousness of the starting variant
- $\sigma_{v}$ is the infectiousness of variant $v$ relative to the starting variant
- $W_{t}$ is the non-mechanistic residual transmission process, under which a Wiener sequence of updates from a starting value of zero is linked with a cosine spline at time $t$
- $s_{v,t}$ is the rate of seeding with variant $v$ at time $t$
- $g_{t}$ is $\int_{t}^{t+1} g(u)\,du$, where $g$ is the probability density function of the generation interval distribution
- $\tilde{v}$ is the variant for which infection results in transition from immunity status $\tilde{p}$ to immunity status $p$
- $G_{l,t}$ is the Google mobility estimate at location $l$ and time $t$
- $F_{t}$ is the Facebook mobility estimate (either tiles visited or single tile) at time $t$
- $w_{l}$ is a calibrated location-specific weight parameter with support $[0, 1]$
- $w_{p}$ is a calibrated policy-specific weight parameter with support $[0, 1]$
- $f$ is a calibrated floor parameter that defines the lowest level of transmission scaling that can occur when policy responses are maximal
- $o_{t}$ is any one of the incident indicator quantities calculated from the incidence time series, i.e. either deaths, case notifications, hospital admission or ICU admissions
- $c_{t}$ is $\int_{t}^{t+1} c(u)\,du$, where $c$ represents the probability density function of the distribution of the delay from the onset of the infection epiosde to the onset of the incident indicator considered
- $a$ is the proportion of infection episodes resulting in the indicator considered
- $h_{t}$ is either one of the prevalent indicators calculated from the incident indicators, i.e. either hospital occupancy or ICU occupancy
- $d_{t}$ is $\int_{t}^{t+1} d(u)\,du$, where $d$ represents the probability density function of the distribution of the time from the onset of the incident indicator (either hospital or ICU admission) to the end of the prevalent indicator period (either hospital or ICU discharge)

In [ ]:
txt = "\n\n## Renewal model\n"
model_rationale = (
    "We developed a discrete renewal model "
    "with daily innovations. The core aspect of our "
    "approach was the convolution of the preceding "
    "strain-specific incidence rate with "
    "the generation interval for subsequent infections, "
    "adjusted for past immunity. "
    "We then applied convolutions to this model "
    "to estimate model outputs that could be compared "
    "against our indicator targets introduced above.\n\n"
)
txt += model_rationale
dummy_start = datetime(2020, 1, 1)
dummy_end = datetime(2021, 12, 31)
no_mob_provider = NoMobilityProvider()
dummy_model = MultiStrainModel(1.0, dummy_start, dummy_end, [""], dummy_start, no_mob_provider, False)
txt += get_func_blurb(dummy_model.renew)
dummy_dist = GammaDens()
txt += get_func_blurb(dummy_dist.get_params) + "\n\n"
txt += "\n\n### Outputs\n"
txt += get_func_blurb(dummy_model.renewal_func) + "\n\n"

txt += "## Modelled population size \n\n"
txt += get_func_blurb(get_country_pop)
txt += get_func_blurb(get_worldbank_national_pop)
txt += get_func_blurb(get_undesa_national_pop)
txt += "\n\n## Analysis period\n"
analysis_period_rationale = (
    "For all countries, we aimed to place our analysis period "
    "during a time period for which the variation in transmission rate "
    "may have been substantially attributable to changes in mobility, "
    "although we developed an approach (described below) to address "
    "the consideration that the emergence of new SARS-CoV-2 variants "
    "likely contributed substantially. "
    "Therefore, we aimed to select time periods during which "
    "the roll-out of vaccination programs would have contributed "
    "less to variations in transmission rate.\n\n"
    "For all countries other than Oceania and Singapore, we addressed this "
    "by limiting our analysis period to the time prior to "
    "vaccination reaching levels that may have had a significant "
    "population-level effect.\n\n"
    "For most countries, a single base case no mobility analysis was run, "
    "against which both the Google and Facebook analyses could be compared. "
    "However, for Oceania and Singapore, the analysis period for the Facebook "
    "analysis was shorter due to the discontinuation of Facebook "
    "data after May 2022. For these countries, an additional "
    "no mobility comparison analysis was run. "
)
txt += analysis_period_rationale
txt += get_func_blurb(find_run_start_time) + "\n\n"
txt += get_func_blurb(find_run_end_time)
txt += get_func_blurb(get_country_vacc_data)
txt += "\n\n## Mobility\n"
mobility_rationale = (
    "The main purpose of our analysis was to understand "
    "the effect of empirically observed changes in mobility "
    "on the observed fluctuations of the COVID-19 epidemic. "
    "We addressed this by including various mobility observation "
    "streams available from Big Tech companies within our model.\n\n"
)
txt += mobility_rationale
txt += get_func_blurb(get_mobility_provider)
txt += "\n\n### No mobility\n"
txt += get_func_blurb(NoMobilityProvider.__init__)
txt += "\n\n### Google mobility\n"
n_domains = 6
dummy_g_mob = pd.DataFrame(1.0, index=range(10), columns=range(n_domains))
dummy_g_priors = {"mob_weights": dist.Uniform(np.zeros(n_domains), np.ones(n_domains)), "mob_exp": None}
weighted_exp_mob_provider = WeightedExpMobilityProvider(dummy_g_mob, dummy_g_priors)
txt += get_func_blurb(process_raw_google_mobility)
txt += get_func_blurb(get_google_mobility)
txt += get_func_blurb(weighted_exp_mob_provider.__init__)
txt += get_func_blurb(weighted_exp_mob_provider.get_parameterised_mobility)
txt += "\n\n### Facebook mobility\n"
dummy_fb_mob_builder = FacebookMobilityBuilder()
txt += get_func_blurb(dummy_fb_mob_builder.build_mobility) + "\n\n"
txt += get_func_blurb(get_fb_visited_mobility)
txt += get_func_blurb(get_fb_singletile_mobility) + "\n\n"
dummy_fb_mob = pd.Series(1.0, index=range(10))
dummy_fb_priors = {"mob_exp": None}
single_series_mob_provider = SingleSeriesExpMobilityProvider(dummy_fb_mob, dummy_fb_priors)
txt += get_func_blurb(single_series_mob_provider.get_parameterised_mobility)
txt += get_func_blurb(single_series_mob_provider.__init__)
txt += "\n\n## Residual transmission process\n"
process_rationale = (
    "We included a non-mechanistic residual process "
    "within our model, which was intended to capture "
    "any variations in transmission that could not be "
    "attributed to the explicitly modelled processes "
    "(variant strain emergence, changes in mobility "
    "and the development of population immunity).\n\n"
)
txt += get_func_blurb(dummy_model.fit_process_curve)
txt += get_func_blurb(_get_cos_curve_at_x)
txt += get_func_blurb(dummy_model.initialise_var_proc)
txt += "\n\n## Calibration targets\n"
indicator_rationale = (
    "We only included countries for which multiple "
    "data streams were available to estimate variations "
    "in transmission rates over time. "
    "Specifically, we required that a time series for both "
    "COVID-19-attributable deaths and case notifications were available. "
    "However, we also included one time series for a hospital-related "
    "indicator, where this could be sourced. "
    "Because we considered that the emergence of new "
    "SARS-CoV-2 variants could have substantially affected transmission, "
    "we included explicit calibration targets for the "
    "strain-specific proportional prevalence of variants where available. "
    "Although it may have been less epidemiologically relevant to "
    "this analysis, we sought to broadly capture the "
    "overall size of the epidemic. This was addressed by "
    "explicitly incorporating susceptible depletion and "
    "including seroprevalence estimates as calibration targets where available. "
    "We did not include vaccination explicitly within the model "
    "because our approach to country and time period inclusion "
    "was intended to focus on time periods during which "
    "vaccination would not have substantially influenced transmission rates.\n\n"
)
txt += indicator_rationale
txt += "\n\n### WHO indicators\n"
txt += get_func_blurb(get_deaths_target)
txt += get_func_blurb(get_cases_target)
txt += "\n\n### Hospitalisations\n"
txt += get_func_blurb(get_owid_hosps)
txt += get_func_blurb(get_hosp_target) + "\n\n"
txt += "\n\n### Seroprevalence\n"
txt += get_func_blurb(get_all_seroprev)
txt += get_func_blurb(filter_seroprev)
txt += get_func_blurb(get_seroprev_pooled_totals) + "\n\n"
txt += get_func_blurb(get_seroprev_target) + "\n\n"
txt += get_func_blurb(get_income_group)
txt += "\n\n### Variants\n\n"
variant_rationale = (
    "We included up to three key variants explicitly within "
    "our analysis framework. For all countries "
    "other than Oceania and Singapore "
    "our approach was intended to represent strains prior to the "
    "emergence of the Alpha variant and strains of Alpha "
    "with strains of Delta also included where sufficient data "
    "were available and the time period simulated meant that "
    "the emergence of Delta was relevant to the simulation. "
    "For Oceania and Singapore, we included pre-BA.2 strains of Omicron, "
    "along with Omicron BA.2 and Omicron BA.5. \n\n"
)
txt += "#### Data extraction\n"
txt += get_func_blurb(get_country_vars)
txt += get_func_blurb(extract_specific_var)
txt += get_func_blurb(get_specific_var_props) + "\n\n"
txt += get_func_blurb(get_var_target)
txt += "\n\n#### Alpha\n"
txt += get_func_blurb(get_alpha_info)
txt += "\n\n#### Delta\n"
txt += get_func_blurb(get_delta_info)
txt += "\n\n#### Omicron BA.2\n"
txt += get_func_blurb(get_ba2_info)
txt += "\n\n#### Omicron BA.5\n"
txt += get_func_blurb(get_ba5_info)

In [ ]:
display(Markdown(txt))

In [ ]:
txt = "\n\n## Calibration algorithm\n"
calibration_rationale = (
    "We used a gradient-based algorithm "
    "for model calibration, which efficiently "
    "explored our multidimensional parameter space. "
    "By exploring all time series in log-space "
    "with a common dispersion parameter, "
    "we were able to apply an algorithm that "
    "provided epidemiologically plausible results for all "
    "countries simulated without the need for "
    "extensive country-specific adaptations.\n\n"
)
txt += calibration_rationale
txt += "\n\n### Initialisation\n"
txt += get_func_blurb(custom_init)
txt += "\n\n### Parameter processing\n"
dummy_calib = StandardCalib(dummy_model, {}, {})
txt += get_func_blurb(dummy_calib.sample_calib_params)
txt += get_func_blurb(dummy_calib.__init__)
txt += "\n\n"
txt += get_func_blurb(get_standard_priors)
txt += "\n\n### Algorithm\n"
txt += get_func_blurb(run_calibration)

In [ ]:
display(Markdown(txt))

In [ ]:
txt = "# Selection of countries for inclusion\n"
either_mob_avail, summary, g_avail, fb_avail = get_mob_avail_countries()
txt += get_func_blurb(get_mob_avail_countries)
death_data, case_data = gather_who_data(either_mob_avail)
txt += get_func_blurb(gather_who_data) + "\n\n"
no_deaths, no_cases = find_absent_inds(death_data, case_data, summary)
txt += get_func_blurb(find_absent_inds)
neg_deaths, neg_cases = find_neg_inds(death_data, case_data, summary)
txt += get_func_blurb(find_neg_inds)
death_outliers, case_outliers = find_outliers(death_data, case_data, summary)
death_nans, case_nans, death_repeats, case_repeats = find_nans_repeats(death_data, case_data, summary)
txt += get_func_blurb(find_nans_repeats) + "\n\n"
txt += "The final results of the selection are presented in @tbl-inc. "
txt += "The final included countries are illustrated in @fig-inc below. "
display(Markdown(txt))

In [ ]:
excluded = set(no_deaths + no_cases + neg_deaths + neg_cases + death_nans + case_nans + death_repeats + case_repeats + death_outliers + case_outliers)
included = [c for c in either_mob_avail if c not in excluded]
add_bool_row_to_table(summary, included, "Incl-uded")

In [ ]:
#| label: fig-inc
#| fig-cap: Included countries and mobility availability. Countries coloured according to mobility availability. Neither source available, grey; Google mobility only available, green; Facebook mobility only available, blue; both sources available, purple. Red markers indicate included in analysis.
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1)
add_mob_avail_to_world(world, g_avail, fb_avail)
world["included"] = world["ISO_A3"].isin(included)
plot_inclusion(world);

In [ ]:
#| label: tbl-inc
#| tbl-cap: Reasons for country inclusion and exclusion.
summary.index = summary.index.map(lambda iso3: pycountry.countries.lookup(iso3).name)
display(Markdown(summary.sort_index().to_markdown()))

\newpage
# Parameter choices and supporting evidence

In [ ]:
loaded_priors = yml.safe_load(open(BASE_PATH / "data/evidence/priors.yml", "r"))
duration_params = loaded_priors["durations"]

## Time period parameters

In [ ]:
col_widths = '{tbl-colwidths="[12, 7, 7, 74]"}'
durations_df = get_table_df_from_priors_dict(loaded_priors["durations"])
keep_cols = [c for c in durations_df if c != "Short_name"]
dur_table = durations_df[keep_cols]
caption = ": Parameters and supporting evidence to time period priors. "
Markdown(dur_table.to_markdown() + "\n" + caption + col_widths)

In [ ]:
#| fig-cap: Duration-related parameter prior distributions.
plot_duration_params(duration_params)

{{< pagebreak >}}

## Proportion parameters

In [ ]:
betas_df = get_table_df_from_priors_dict(loaded_priors["beta"])
caption = ": Parameters and supporting evidence to beta-distributed priors. "
Markdown(betas_df.to_markdown() + "\n" + caption + col_widths)

In [ ]:
#| fig-cap: Proportion parameter prior distributions. Note horizontal axis range differs between upper two panels and lower three panels.
plot_beta_priors(loaded_priors["beta"])

In [ ]:
fixed_params = loaded_priors["fixed"]
params_df = pd.DataFrame.from_dict(fixed_params).T
params_df = params_df.set_index("param_name")
params_df.columns = params_df.columns.str.capitalize()
params_df.index.name = None
caption = "\n: Fixed parameters and supporting evidence. "

{{< pagebreak >}}

## Fixed parameters

In [ ]:
Markdown(params_df.to_markdown() + caption + '{tbl-colwidths="[12, 7, 81]"}')

{{< pagebreak >}}

## Relative infectiousness parameter

In [ ]:
Markdown(loaded_priors["other"]["relinfect"]["evidence"])

{{< pagebreak >}}

# References